# ISOM 835 · Session 12 — Neural Networks, Embeddings & LLMs as Features
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Dec 7 · Prof. Hasan Arslan**

A multilayer perceptron on tabular data (and why it usually loses to boosting), text as features three ways — TF-IDF, sentence embeddings, zero-shot LLM labels — embeddings inside a tabular model, and the AI-assisted analyst's workflow: *generate, verify, own*.

In [ ]:
import pandas as pd, numpy as np, time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

## 1. A neural network on a business table — the honest result
A logistic regression is a neural network with no hidden layer. `MLPClassifier` adds layers of weighted sums and nonlinearities, trained by gradient descent. On 7,000 rows of Telco it ties logistic regression and trails boosting — the Grinsztajn result on a business table.

In [ ]:
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(URL); df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
X = pd.get_dummies(df.drop(columns=['customerID', 'Churn']), drop_first=True).astype(float); y = (df['Churn'] == 'Yes').astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)
for name, m in [('logistic', make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))),
                ('MLP 64-32', make_pipeline(StandardScaler(), MLPClassifier((64, 32), early_stopping=True, max_iter=500, random_state=835))),
                ('gradient boosting', HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, early_stopping=True, random_state=835))]:
    t0 = time.time(); m.fit(X_tr, y_tr); print(f'{name:18s} test AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.3f}   ({time.time()-t0:.1f}s)')

Neural nets earn their keep on text, images, audio, sequences — anything without fixed columns. Their real gift to tabular work is **embeddings**.

## 2. Text as features I — TF-IDF
Amazon product reviews, positive vs. negative. Counts of words and bigrams, weighted by rarity: sparse, fast, and you can read the coefficients.

In [ ]:
# Load 5,000 reviews (Hugging Face `datasets`, no login). Fallback: 20 newsgroups if offline.
try:
    from datasets import load_dataset
    ds = load_dataset('fancyzhx/amazon_polarity', split='train[:5000]')
    texts, labels = list(ds['content']), np.array(ds['label']); names = ['negative', 'positive']
except Exception as e:
    print('datasets unavailable →', type(e).__name__, '— falling back to 20 newsgroups (rec.autos vs sci.med)')
    from sklearn.datasets import fetch_20newsgroups
    ng = fetch_20newsgroups(subset='train', categories=['rec.autos', 'sci.med'], remove=('headers', 'footers', 'quotes'))
    texts, labels, names = ng.data, ng.target, ng.target_names
T_tr, T_te, l_tr, l_te = train_test_split(texts, labels, test_size=0.2, stratify=labels, random_state=835)
print(len(T_tr), 'train texts;', names)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
clf = LogisticRegression(max_iter=3000, C=3).fit(tfidf.fit_transform(T_tr), l_tr)
print(f'TF-IDF + logistic: accuracy {accuracy_score(l_te, clf.predict(tfidf.transform(T_te))):.3f}   ({len(tfidf.vocabulary_):,} features)')
vocab = np.array(tfidf.get_feature_names_out()); w = clf.coef_[0]
print('most', names[1], ':', ', '.join(vocab[np.argsort(-w)[:8]])); print('most', names[0], ':', ', '.join(vocab[np.argsort(w)[:8]]))

## 3. Text as features II — sentence embeddings
A pretrained sentence-transformer turns each review into a 384-number vector that captures *meaning* ("not bad at all" ≈ positive). Then any model you already know.

In [ ]:
# OPTIONAL — pip install sentence-transformers   (downloads a ~90MB model once)
try:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer('all-MiniLM-L6-v2')
    E_tr = enc.encode(T_tr, batch_size=128, show_progress_bar=False); E_te = enc.encode(T_te, batch_size=128, show_progress_bar=False)
    clf2 = LogisticRegression(max_iter=3000).fit(E_tr, l_tr)
    print(f'embeddings + logistic: accuracy {accuracy_score(l_te, clf2.predict(E_te)):.3f}   ({E_tr.shape[1]} features)')
except ImportError:
    print('sentence-transformers not installed')

## 4. Text as features III — zero-shot LLM labels (optional, needs an API key)
Ask a language model to label 100 test reviews with no training data. Competitive on small, simple tasks; costs a call per row; the wrong tool for a million reviews a day.

In [ ]:
# OPTIONAL — pip install anthropic ; set ANTHROPIC_API_KEY in Colab secrets (🔑 panel) or os.environ
import os
if os.environ.get('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic(); preds = []
    for t in T_te[:100]:
        msg = client.messages.create(model='claude-sonnet-5', max_tokens=5,
            messages=[{'role': 'user', 'content': f"Classify this product review as exactly one word, '{names[0]}' or '{names[1]}':\n\n{t[:1500]}"}])
        preds.append(1 if names[1] in msg.content[0].text.lower() else 0)
    print(f'zero-shot LLM: accuracy on 100 reviews {accuracy_score(l_te[:100], preds):.3f}')
else:
    print('no ANTHROPIC_API_KEY set — skipping the zero-shot cell')

## 5. Embeddings inside a tabular model
The practical 2026 pattern: embed a free-text column, append the vector (or its PCA) to the structured features, and let boosting use both. Helps most when the table is small or the text carries what the columns don't. Here: a synthetic "reason for contacting support" note attached to Telco customers.

In [ ]:
rng = np.random.default_rng(835)
notes_churn = ['thinking of switching providers', 'bill keeps going up, unhappy', 'internet drops every evening', 'want to cancel my contract', 'competitor offered a better deal']
notes_stay = ['question about my invoice', 'add a streaming package', 'moving, transfer service', 'set up autopay', 'upgrade router please']
note = np.where(rng.random(len(df)) < 0.6, np.where(y == 1, rng.choice(notes_churn, len(df)), rng.choice(notes_stay, len(df))), rng.choice(notes_stay + notes_churn, len(df)))
try:
    Emb = enc.encode(list(note), batch_size=256, show_progress_bar=False)
except NameError:
    from sklearn.feature_extraction.text import TfidfVectorizer as _T; from sklearn.decomposition import TruncatedSVD
    Emb = TruncatedSVD(16, random_state=835).fit_transform(_T().fit_transform(note))          # fallback: TF-IDF + SVD "embedding"
Xe = pd.concat([X.reset_index(drop=True), pd.DataFrame(Emb, columns=[f'emb{i}' for i in range(Emb.shape[1])])], axis=1)
Xe_tr, Xe_te = Xe.loc[X_tr.index], Xe.loc[X_te.index]
for name, (A, B) in [('structured only', (X_tr, X_te)), ('+ text embedding', (Xe_tr, Xe_te))]:
    m = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, early_stopping=True, random_state=835).fit(A, y_tr)
    print(f'{name:18s} test AUC {roc_auc_score(y_te, m.predict_proba(B)[:, 1]):.3f}')

## 6. The AI-assisted analyst: generate, verify, own
Colab's Data Science Agent, Jupyter AI, Claude Code can draft a whole notebook. Your job shifts from typing to judging. **Verify the four things that matter in any AI-written notebook:**

| Check | Question | Where it fails |
|---|---|---|
| **The split** | Is every fitted transformer inside a pipeline fit on train only? | scaler/encoder fit before `train_test_split` |
| **The leakage** | Is any feature recorded after the outcome? | `duration`, `reservation_status`, post-event flags |
| **The metric** | Does the metric match the decision and the base rate? | accuracy on a 2% positive class |
| **The baseline** | Does it beat the obvious rule? | no majority-class / seasonal-naive comparison |

Then the presentation test: can you explain every cell without the assistant?

In [ ]:
# A checklist you can run on any notebook: find fitted transformers that are called before a split
import re, json, sys
def audit(path):
    nb = json.load(open(path)); code = '\n'.join(''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code')
    split_at = code.find('train_test_split('); fits = [m.start() for m in re.finditer(r'\.fit_transform\(|\.fit\(', code)]
    early = [f for f in fits if split_at == -1 or f < split_at]
    print(f'{path}: {len(fits)} fit calls, {len(early)} before the first train_test_split → {"CHECK THESE" if early else "ok"}')
    for kw in ['duration', 'reservation_status', 'accuracy_score']: print(f"  mentions '{kw}':", kw in code)
audit('ISOM835_Session12_NeuralNets_Embeddings_LLMs.ipynb') if __import__('os').path.exists('ISOM835_Session12_NeuralNets_Embeddings_LLMs.ipynb') else print('run inside the notebook folder to audit')

## 7. Your turn
1. **Depth.** Try MLP sizes (16,), (128, 64), (256, 128, 64) on Telco. Does more capacity help? Compare with the boosting line.
2. **Embedding + tabular on the real column.** Telco has no text — but hotel bookings has `country` and `market_segment`. Embed the *category names* with the sentence-transformer and check whether boosting gains anything over one-hot. (It usually won't. Say why.)
3. **Audit a friend's notebook** with the checklist function above. Report what it flagged and whether the flag was a real problem.

In [ ]:
# Your turn — work here

## What we learned tonight
- A neural network is **stacked weighted sums with nonlinearities**; on business tables it rarely beats boosting — it wins on text, images, sequences.
- **Text becomes features three ways:** TF-IDF (counts, interpretable), embeddings (meaning, transferable), zero-shot LLM labels (judgment, per-call cost). Embeddings appended to a tabular model are the practical pattern.
- **Generate, verify, own.** The split, the leakage, the metric, the baseline — check them yourself.

Project notebook + memo due **Sun Dec 13, 11:59 PM**. Presentations **Mon Dec 14**.